# AI4002 — Computer Vision | Semester Project Part 2 (Model 2: Runs 4–6)
## Hybrid Feature Learning: ConvNeXt-Base + Cross-Attention Handcrafted Feature Fusion on CheXpert

### 1. Project Information

| Field | Details |
|---|---|
| **Group No.** | `[GROUP NUMBER]` |
| **Student Name** | `[YOUR FULL NAME]` |
| **Roll No.** | `[YOUR ROLL NUMBER]` |
| **Project Title** | Hybrid Feature Learning: Integrating Handcrafted Visual Descriptors with Advanced Vision Models for Multi-Label Chest X-Ray Classification |
| **Base Paper** | Baharoon et al. (2024), *Evaluating General Purpose Vision Foundation Models for Medical Image Analysis: An Experimental Study of DINOv2 on Radiology Benchmarks* (`arXiv:2312.02366v4`) |
| **Dataset** | CheXpert (5 competition pathologies, frontal views, 10,000-sample subset + fixed official val + deterministic held-out test — **identical to Notebook 08**) |
| **Advanced Model 2** | **ConvNeXt-Base** (Hierarchical Modern CNN with 7×7 depthwise convolutions — distinct architecture family from Model 1's DINOv2 ViT-B/14) |
| **Feature Extractor 3** | **Multi-Scale LBP (Local Binary Patterns)** — 432-dim circular micro-texture descriptor across radii R ∈ {1, 2, 3} |
| **Feature Extractor 4** | **Multi-Scale Edge & Laplacian Pyramid (Sobel + LoG + Intensity Stats)** — 192-dim structural & blob-contour descriptor |
| **Feature Fusion Strategy 2** | **Multi-Head Cross-Attention Feature Fusion** (deep ConvNeXt features attend to projected handcrafted visual tokens) |

### Experimental Matrix — Model 2 (Runs 4–6 of 6; Complements `08_hybrid_fusion_chexpert.ipynb` Runs 1–3)

| Run | Configuration | Description |
|---|---|---|
| **Run 4 (Model 2 Run 1)** | `ConvNeXt-Base` (Deep Only) | Train & evaluate ConvNeXt-Base **without** additional handcrafted features |
| **Run 5 (Model 2 Run 2)** | `ConvNeXt-Base + Multi-Scale LBP` | Integrate ConvNeXt-Base with **Multi-Scale LBP** via Cross-Attention Feature Fusion |
| **Run 6 (Model 2 Run 3)** | `ConvNeXt-Base + Edge/LoG Pyramid` | Integrate ConvNeXt-Base with **Multi-Scale Edge/LoG Pyramid** via Cross-Attention Feature Fusion |

In [ ]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if os.path.basename(os.getcwd()) != "DinoV2_PEFT":
    if not os.path.exists("DinoV2_PEFT"):
        subprocess.run(["git", "clone", "https://github.com/RudhanLodhi/DinoV2_PEFT.git"], check=True)
    os.chdir("DinoV2_PEFT")

subprocess.run(["pip", "install", "-r", "requirements.txt", "-q"], check=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft/part2_chexpert"
except ImportError:
    CHECKPOINT_DIR = "outputs/part2_chexpert"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Working directory:    {os.getcwd()}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 2. Experimental Consistency & Fixed Configuration (Identical to Model 1)

Matches Notebook `08_hybrid_fusion_chexpert.ipynb` identically on dataset split (`seed=42`), preprocessing/augmentation, `epochs=10`, `batch_size=32`, `AdamW`, Warmup + Cosine LR schedule, `pos_weight`, and all 4 evaluation metrics so all **6 runs** can be compared fairly.

In [ ]:
"""
Load CheXpert configuration and lock shared Part 2 hyperparameters (identical split/epochs/BS to Model 1).
"""
import json
import random
import time
from typing import Any
import numpy as np
import torch
from src.utils.config import load_config

cfg: dict[str, Any] = load_config("configs/chexpert.yaml")

PART2_M2_CFG: dict[str, Any] = {
    "seed": 42,
    "epochs": 10,
    "batch_size": 32,
    "backbone_lr": 1e-4,
    "head_lr": 1e-3,
    "weight_decay": 1e-4,
    "proj_dim": 256,
    "dropout": 0.1,
    "unfreeze_last_stages": 2,
}

cfg["training"]["batch_size"] = PART2_M2_CFG["batch_size"]
LABEL_NAMES: list[str] = list(cfg["dataset"]["label_names"])
NUM_LABELS: int = len(LABEL_NAMES)
BASE_PAPER_CHEXPERT_AUROC: float = 0.812

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | Labels ({NUM_LABELS}): {LABEL_NAMES}")
print("Fixed Part 2 Model 2 Hyperparameters:", json.dumps(PART2_M2_CFG, indent=2))

## 3. Build Identical CheXpert DataLoaders & Class Imbalance Weights

In [ ]:
"""
Construct identical CheXpert DataLoaders (same seed=42 split as Model 1) and per-label pos_weight.
"""
from src.data import build_chexpert_dataloaders, build_transforms

set_seed(PART2_M2_CFG["seed"])
pre = cfg["preprocessing"]
train_tf = build_transforms(
    pre["image_size"],
    pre["normalize_mean"],
    pre["normalize_std"],
    augment=True,
    crop_scale_min=pre.get("crop_scale_min", 0.75),
    crop_scale_max=pre.get("crop_scale_max", 1.0),
)
val_tf = build_transforms(
    pre["image_size"],
    pre["normalize_mean"],
    pre["normalize_std"],
    augment=False,
    resize_ratio=pre.get("resize_ratio", 1.14),
)

train_loader, val_loader, test_loader = build_chexpert_dataloaders(cfg, train_tf, val_tf)

pos_counts = torch.zeros(NUM_LABELS, dtype=torch.float64)
total_n = 0
for b in train_loader:
    pos_counts += b["labels"].sum(dim=0).double()
    total_n += b["labels"].shape[0]
POS_WEIGHT = ((total_n - pos_counts) / pos_counts.clamp(min=1.0)).clamp(1.0, 5.0).float()

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")
print("Per-label pos_weight:", dict(zip(LABEL_NAMES, [round(float(w), 2) for w in POS_WEIGHT])))

## 4. Handcrafted Feature Extractors for Model 2 (Multi-Scale LBP & Edge/LoG Pyramid)

1. **Multi-Scale LBP (`MultiScaleLBPExtractor`):** Captures fine-grained lung parenchymal micro-textures (interstitial edema, reticular opacities, alveolar consolidation) across radii `R ∈ {1, 2, 3}` (432-dim).
2. **Multi-Scale Edge & Laplacian Pyramid (`EdgeLaplacianPyramidExtractor`):** Captures Sobel gradient boundaries, Laplacian-of-Gaussian (LoG) blob/opacity responses, and regional intensity statistics across 3 spatial scales (192-dim).

In [ ]:
"""
Handcrafted Feature Extractors for Advanced Model 2 (GPU-vectorized pure PyTorch):
  1. MultiScaleLBPExtractor     -- Multi-radius (R=1, 2, 3) Local Binary Patterns (micro-texture)
  2. EdgeLaplacianPyramidExtractor -- Multi-scale Sobel Edge + Laplacian-of-Gaussian + Intensity Stats

Both operate directly on ImageNet-normalized (B, 3, 224, 224) batches on GPU.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiScaleLBPExtractor(nn.Module):
    """Multi-radius Local Binary Pattern (LBP) descriptor over radii R in (1, 2, 3).

    For each radius R, compares each pixel against 8 circularly sampled neighbors to form
    8-bit local micro-texture response maps, then pools spatial 4x4 grid statistics
    (mean & std of bit-activation energy + transition count / uniformity energy).
    Output dimension: len(radii) * 9 * 16 = 3 * 144 = 432.
    """

    def __init__(self, radii: tuple[int, ...] = (1, 2, 3), grid_size: int = 4) -> None:
        super().__init__()
        self.radii = radii
        self.grid_size = grid_size
        self.out_dim = len(radii) * 9 * (grid_size * grid_size)

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        # Precompute 8 directional difference kernels for each radius
        self.num_neighbors = 8
        for r in radii:
            ksize = 2 * r + 1
            kernels = torch.zeros(8, 1, ksize, ksize, dtype=torch.float32)
            for idx in range(8):
                ang = 2.0 * math.pi * idx / 8.0
                dy = int(round(r * math.sin(ang)))
                dx = int(round(r * math.cos(ang)))
                kernels[idx, 0, r + dy, r + dx] = 1.0
                kernels[idx, 0, r, r] = -1.0
            self.register_buffer(f"lbp_k_{r}", kernels)

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_raw = (x.float() * self.std + self.mean).clamp(0.0, 1.0)
        gray = 0.299 * x_raw[:, 0:1] + 0.587 * x_raw[:, 1:2] + 0.114 * x_raw[:, 2:3]

        feats = []
        for r in self.radii:
            k = getattr(self, f"lbp_k_{r}")
            diffs = F.conv2d(gray, k, padding=r)  # (B, 8, H, W)
            bits = (diffs >= 0).float()  # binary neighbor indicators
            # Uniformity transitions between adjacent circular bits (captures smooth vs. noisy texture)
            transitions = torch.abs(bits - torch.roll(bits, shifts=1, dims=1)).mean(dim=1, keepdim=True)
            stacked = torch.cat([bits, transitions], dim=1)  # (B, 9, H, W)
            pooled = F.adaptive_avg_pool2d(stacked, (self.grid_size, self.grid_size)).flatten(start_dim=1)
            feats.append(pooled)

        out = torch.cat(feats, dim=-1)
        return F.normalize(out, p=2, dim=-1, eps=1e-6)


class EdgeLaplacianPyramidExtractor(nn.Module):
    """Multi-scale Edge (Sobel) + Laplacian-of-Gaussian (LoG) + Intensity Statistics descriptor.

    Evaluates 3 pyramid scales (full 224, half 112, quarter 56) extracting Sobel edge energy,
    LoG blob/contour energy, and local intensity mean/std over a 4x4 spatial grid.
    Output dimension: 3 scales * 4 channels * 16 grid cells = 192.
    """

    def __init__(self, grid_size: int = 4) -> None:
        super().__init__()
        self.grid_size = grid_size
        self.out_dim = 3 * 4 * (grid_size * grid_size)

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
        sobel_y = torch.tensor([[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]])
        log_k = torch.tensor(
            [
                [0.0, 0.0, -1.0, 0.0, 0.0],
                [0.0, -1.0, -2.0, -1.0, 0.0],
                [-1.0, -2.0, 16.0, -2.0, -1.0],
                [0.0, -1.0, -2.0, -1.0, 0.0],
                [0.0, 0.0, -1.0, 0.0, 0.0],
            ]
        ) / 16.0
        self.register_buffer("sobel_x", sobel_x.view(1, 1, 3, 3))
        self.register_buffer("sobel_y", sobel_y.view(1, 1, 3, 3))
        self.register_buffer("log_k", log_k.view(1, 1, 5, 5))

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_raw = (x.float() * self.std + self.mean).clamp(0.0, 1.0)
        gray = 0.299 * x_raw[:, 0:1] + 0.587 * x_raw[:, 1:2] + 0.114 * x_raw[:, 2:3]

        feats = []
        cur = gray
        for s in range(3):
            if s > 0:
                cur = F.avg_pool2d(cur, kernel_size=2, stride=2)
            gx = F.conv2d(cur, self.sobel_x, padding=1)
            gy = F.conv2d(cur, self.sobel_y, padding=1)
            edge_mag = torch.sqrt(gx * gx + gy * gy + 1e-8)
            log_resp = torch.abs(F.conv2d(cur, self.log_k, padding=2))
            local_mean = F.adaptive_avg_pool2d(cur, (self.grid_size, self.grid_size))
            local_sq = F.adaptive_avg_pool2d(cur * cur, (self.grid_size, self.grid_size))
            local_std = torch.sqrt((local_sq - local_mean * local_mean).clamp(min=1e-8))
            edge_pool = F.adaptive_avg_pool2d(edge_mag, (self.grid_size, self.grid_size))
            log_pool = F.adaptive_avg_pool2d(log_resp, (self.grid_size, self.grid_size))
            feats.append(torch.cat([local_mean, local_std, edge_pool, log_pool], dim=1).flatten(start_dim=1))

        out = torch.cat(feats, dim=-1)
        return F.normalize(out, p=2, dim=-1, eps=1e-6)


_dummy = torch.randn(2, 3, 224, 224)
_lbp = MultiScaleLBPExtractor()
_edge = EdgeLaplacianPyramidExtractor()
print(f"Multi-Scale LBP output shape:          {tuple(_lbp(_dummy).shape)} (dim={_lbp.out_dim})")
print(f"Edge + Laplacian Pyramid output shape: {tuple(_edge(_dummy).shape)} (dim={_edge.out_dim})")

## 5. Advanced Model 2 (ConvNeXt-Base) & Cross-Attention Feature Fusion Architecture

In [ ]:
"""
Advanced Model 2 (ConvNeXt-Base) + Cross-Attention Feature Fusion Architecture.

Why ConvNeXt-Base as Advanced Model 2 (distinct from Model 1 DINOv2 ViT-B/14 + LoRA):
  - Hierarchical pure-CNN architecture with 7x7 depthwise convolutions and inverted bottlenecks.
  - Strong local translation equivariance and multi-scale inductive bias complementary to
    DINOv2's isotropic global self-attention.
  - Pretrained on ImageNet-1K/22K; we fine-tune the upper stages (stages 2-3 + final norm)
    for parameter-efficient medical adaptation while freezing early generic edge filters.

Feature Fusion Strategy 2 (Attention-Based Feature Fusion — distinct from Model 1's Gated Concatenation):
  - Projects ConvNeXt pooled features (1024 -> 256) as Query token Q (B, 1, 256).
  - Splits & projects handcrafted features into K=4 Key/Value visual tokens (B, 4, 256).
  - Applies 4-head Cross-Attention (MultiheadAttention) with residual skip connection + LayerNorm,
    so the deep CNN representation dynamically attends to the most informative handcrafted cues.
"""
import torch
import torch.nn as nn
from torchvision.models import convnext_base, ConvNeXt_Base_Weights


def build_convnext_backbone(unfreeze_last_stages: int = 2) -> tuple[nn.Module, int]:
    """Loads pretrained ConvNeXt-Base and unfreezes the top `unfreeze_last_stages` stages."""
    model = convnext_base(weights=ConvNeXt_Base_Weights.DEFAULT)
    # Freeze all parameters first
    for p in model.parameters():
        p.requires_grad = False
    # ConvNeXt features has 8 blocks: [stem(0), stage0(1), down(2), stage1(3), down(4), stage2(5), down(6), stage3(7)]
    unfreeze_from = max(0, len(model.features) - (unfreeze_last_stages * 2))
    for idx in range(unfreeze_from, len(model.features)):
        for p in model.features[idx].parameters():
            p.requires_grad = True
    # Remove stock classifier linear layer, keep LayerNorm + Flatten
    feat_dim = model.classifier[2].in_features  # 1024
    model.classifier[2] = nn.Identity()
    return model, feat_dim


class CrossAttentionConvNeXtModel(nn.Module):
    """ConvNeXt-Base with optional Multi-Head Cross-Attention Handcrafted Feature Fusion."""

    def __init__(
        self,
        backbone: nn.Module,
        feat_dim: int,
        num_outputs: int,
        feature_extractor: nn.Module | None = None,
        proj_dim: int = 256,
        num_hc_tokens: int = 4,
        num_heads: int = 4,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.backbone = backbone
        self.feature_extractor = feature_extractor
        self.proj_dim = proj_dim
        self.num_hc_tokens = num_hc_tokens

        self.deep_proj = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, proj_dim),
            nn.GELU(),
        )

        if feature_extractor is not None:
            hc_dim = int(feature_extractor.out_dim)
            self.hc_tokenizer = nn.Sequential(
                nn.LayerNorm(hc_dim),
                nn.Linear(hc_dim, proj_dim * num_hc_tokens),
                nn.GELU(),
            )
            self.cross_attn = nn.MultiheadAttention(
                embed_dim=proj_dim,
                num_heads=num_heads,
                dropout=dropout,
                batch_first=True,
            )
            self.attn_norm = nn.LayerNorm(proj_dim)
            self.ffn = nn.Sequential(
                nn.Linear(proj_dim, proj_dim * 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(proj_dim * 2, proj_dim),
            )
            self.ffn_norm = nn.LayerNorm(proj_dim)
        else:
            self.hc_tokenizer = None
            self.cross_attn = None

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(proj_dim, num_outputs),
        )

    def head_parameters(self):
        params = list(self.deep_proj.parameters()) + list(self.classifier.parameters())
        if self.hc_tokenizer is not None:
            params += (
                list(self.hc_tokenizer.parameters())
                + list(self.cross_attn.parameters())
                + list(self.attn_norm.parameters())
                + list(self.ffn.parameters())
                + list(self.ffn_norm.parameters())
            )
        return params

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        deep_vec = self.backbone(pixel_values)  # (B, 1024)
        f_deep = self.deep_proj(deep_vec)  # (B, proj_dim)

        if self.feature_extractor is not None:
            hc_raw = self.feature_extractor(pixel_values).to(f_deep.dtype)
            hc_tokens = self.hc_tokenizer(hc_raw).view(-1, self.num_hc_tokens, self.proj_dim)
            q = f_deep.unsqueeze(1)  # (B, 1, proj_dim)
            attn_out, _ = self.cross_attn(query=q, key=hc_tokens, value=hc_tokens)
            fused = self.attn_norm(q + attn_out).squeeze(1)
            fused = self.ffn_norm(fused + self.ffn(fused))
            return self.classifier(fused)

        return self.classifier(f_deep)

## 6. Unified Training & 4-Metric Evaluation Pipeline

In [ ]:
"""
Unified training and 4-metric evaluation runner for Model 2 (Runs 4, 5, 6).
"""
import copy
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.optim import AdamW
from src.evaluation.metrics import compute_auroc
from src.models import count_trainable_parameters
from src.training.losses import MultiLabelBCELoss
from src.training.schedulers import get_cosine_schedule_with_warmup


@torch.no_grad()
def evaluate_multilabel(model: nn.Module, loader, device: torch.device) -> dict[str, Any]:
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        pv = batch["pixel_values"].to(device, non_blocking=True)
        lb = batch["labels"]
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(pv)
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(lb.numpy())

    logits_np = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0).astype(int)
    probs_np = 1.0 / (1.0 + np.exp(-logits_np))
    preds_np = (probs_np >= 0.5).astype(int)

    auroc_out = compute_auroc(labels_np, probs_np, label_names=LABEL_NAMES)
    return {
        "mean_auroc": float(auroc_out["mean_auroc"]),
        "per_label_auroc": {k: float(v) for k, v in auroc_out["per_label_auroc"].items()},
        "macro_f1": float(f1_score(labels_np, preds_np, average="macro", zero_division=0.0)),
        "macro_precision": float(precision_score(labels_np, preds_np, average="macro", zero_division=0.0)),
        "macro_recall": float(recall_score(labels_np, preds_np, average="macro", zero_division=0.0)),
    }


def run_model2_experiment(
    run_id: str,
    run_label: str,
    feature_extractor: nn.Module | None,
) -> dict[str, Any]:
    set_seed(PART2_M2_CFG["seed"])
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    backbone, feat_dim = build_convnext_backbone(
        unfreeze_last_stages=PART2_M2_CFG["unfreeze_last_stages"]
    )
    model = CrossAttentionConvNeXtModel(
        backbone=backbone,
        feat_dim=feat_dim,
        num_outputs=NUM_LABELS,
        feature_extractor=feature_extractor,
        proj_dim=PART2_M2_CFG["proj_dim"],
        dropout=PART2_M2_CFG["dropout"],
    ).to(device)

    trainable_params, total_params = count_trainable_parameters(model)
    print(f"\n================ {run_label} ================")
    print(
        f"Trainable params: {trainable_params:,} / {total_params:,} "
        f"({100.0 * trainable_params / total_params:.2f}%)"
    )

    criterion = MultiLabelBCELoss(pos_weight=POS_WEIGHT.to(device))
    optimizer = AdamW(
        [
            {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": PART2_M2_CFG["backbone_lr"]},
            {"params": model.head_parameters(), "lr": PART2_M2_CFG["head_lr"]},
        ],
        weight_decay=PART2_M2_CFG["weight_decay"],
    )
    total_steps = PART2_M2_CFG["epochs"] * len(train_loader)
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

    history: list[dict[str, float]] = []
    best_val_auroc: float = -1.0
    best_epoch: int = 0
    best_state = None
    start_time = time.time()

    for epoch in range(1, PART2_M2_CFG["epochs"] + 1):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            pv = batch["pixel_values"].to(device, non_blocking=True)
            lb = batch["labels"].to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits = model(pv)
                loss = criterion(logits, lb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        val_metrics = evaluate_multilabel(model, val_loader, device)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_auroc": val_metrics["mean_auroc"],
                "val_macro_f1": val_metrics["macro_f1"],
            }
        )
        print(
            f"Epoch [{epoch:02d}/{PART2_M2_CFG['epochs']:02d}] "
            f"Loss: {train_loss:.4f} | Val AUROC: {val_metrics['mean_auroc']:.4f} | "
            f"Val F1: {val_metrics['macro_f1']:.4f}"
        )
        if val_metrics["mean_auroc"] > best_val_auroc:
            best_val_auroc = val_metrics["mean_auroc"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    elapsed_sec = time.time() - start_time
    peak_mem_mb = (
        torch.cuda.max_memory_allocated(device) / (1024**2) if device.type == "cuda" else 0.0
    )

    model.load_state_dict(best_state)
    best_val = evaluate_multilabel(model, val_loader, device)
    best_test = evaluate_multilabel(model, test_loader, device)

    ckpt_path = f"{CHECKPOINT_DIR}/{run_id}_best.pt"
    torch.save(best_state, ckpt_path)

    print(f"Best Epoch: {best_epoch} | Time: {elapsed_sec:.1f}s | Peak GPU: {peak_mem_mb:.1f} MB")
    print(
        f"Test -> Mean AUROC: {best_test['mean_auroc']:.4f} | Macro-F1: {best_test['macro_f1']:.4f} | "
        f"Precision: {best_test['macro_precision']:.4f} | Recall: {best_test['macro_recall']:.4f}"
    )

    return {
        "run_id": run_id,
        "run_label": run_label,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "best_epoch": best_epoch,
        "train_time_sec": round(elapsed_sec, 1),
        "peak_gpu_mb": round(peak_mem_mb, 1),
        "val": best_val,
        "test": best_test,
        "history": history,
        "checkpoint": ckpt_path,
    }


MODEL2_RUNS: dict[str, dict[str, Any]] = {}

## 7. Run 4 (Model 2 Run 1) — ConvNeXt-Base Without Additional Features

In [ ]:
MODEL2_RUNS["run4_convnext_only"] = run_model2_experiment(
    run_id="run4_convnext_only",
    run_label="Run 4: ConvNeXt-Base (No Extra Features)",
    feature_extractor=None,
)

## 8. Run 5 (Model 2 Run 2) — ConvNeXt-Base + Multi-Scale LBP (Cross-Attention Fusion)

In [ ]:
MODEL2_RUNS["run5_convnext_lbp"] = run_model2_experiment(
    run_id="run5_convnext_lbp",
    run_label="Run 5: ConvNeXt-Base + Multi-Scale LBP (Cross-Attn)",
    feature_extractor=MultiScaleLBPExtractor(radii=(1, 2, 3), grid_size=4),
)

## 9. Run 6 (Model 2 Run 3) — ConvNeXt-Base + Edge/LoG Pyramid (Cross-Attention Fusion)

In [ ]:
MODEL2_RUNS["run6_convnext_edgelog"] = run_model2_experiment(
    run_id="run6_convnext_edgelog",
    run_label="Run 6: ConvNeXt-Base + Edge/LoG Pyramid (Cross-Attn)",
    feature_extractor=EdgeLaplacianPyramidExtractor(grid_size=4),
)

## 10. Model 2 Results + Consolidated All-6-Runs Comparison Table & Plots

Automatically combines Runs 4–6 from this notebook with Runs 1–3 from `08_hybrid_fusion_chexpert.ipynb` (if saved in `CHECKPOINT_DIR`) to generate the complete Part 2 **6-run comparison table**, Part 1 Base Paper delta, and comparative charts.

In [ ]:
"""
Compile Model 2 comparison table, merge with Model 1 (if available) for full 6-run Part 2 report,
and export CSV/JSON + plots.
"""
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

r4 = MODEL2_RUNS["run4_convnext_only"]
r5 = MODEL2_RUNS["run5_convnext_lbp"]
r6 = MODEL2_RUNS["run6_convnext_edgelog"]

m2_table = pd.DataFrame(
    [
        {
            "Metric": "Mean AUROC (Test)",
            "Run 4: ConvNeXt-Base": round(r4["test"]["mean_auroc"], 4),
            "Run 5: ConvNeXt + LBP": round(r5["test"]["mean_auroc"], 4),
            "Run 6: ConvNeXt + Edge/LoG": round(r6["test"]["mean_auroc"], 4),
        },
        {
            "Metric": "Macro-F1 (Test)",
            "Run 4: ConvNeXt-Base": round(r4["test"]["macro_f1"], 4),
            "Run 5: ConvNeXt + LBP": round(r5["test"]["macro_f1"], 4),
            "Run 6: ConvNeXt + Edge/LoG": round(r6["test"]["macro_f1"], 4),
        },
        {
            "Metric": "Macro Precision (Test)",
            "Run 4: ConvNeXt-Base": round(r4["test"]["macro_precision"], 4),
            "Run 5: ConvNeXt + LBP": round(r5["test"]["macro_precision"], 4),
            "Run 6: ConvNeXt + Edge/LoG": round(r6["test"]["macro_precision"], 4),
        },
        {
            "Metric": "Macro Recall (Test)",
            "Run 4: ConvNeXt-Base": round(r4["test"]["macro_recall"], 4),
            "Run 5: ConvNeXt + LBP": round(r5["test"]["macro_recall"], 4),
            "Run 6: ConvNeXt + Edge/LoG": round(r6["test"]["macro_recall"], 4),
        },
        {
            "Metric": "Mean AUROC (Val)",
            "Run 4: ConvNeXt-Base": round(r4["val"]["mean_auroc"], 4),
            "Run 5: ConvNeXt + LBP": round(r5["val"]["mean_auroc"], 4),
            "Run 6: ConvNeXt + Edge/LoG": round(r6["val"]["mean_auroc"], 4),
        },
    ]
)
print("=== Part 2 Model 2 Comparison Table (CheXpert) ===")
print(m2_table.to_string(index=False))

m2_table.to_csv(f"{CHECKPOINT_DIR}/part2_model2_chexpert_comparison.csv", index=False)
with open(f"{CHECKPOINT_DIR}/part2_model2_chexpert_results.json", "w", encoding="utf-8") as f:
    json.dump(MODEL2_RUNS, f, indent=2)

# Merge with Model 1 (Runs 1-3) for complete 6-run table
m1_json = f"{CHECKPOINT_DIR}/part2_chexpert_results.json"
combined_runs = {}
if os.path.exists(m1_json):
    with open(m1_json, "r", encoding="utf-8") as f:
        combined_runs.update(json.load(f))
combined_runs.update(MODEL2_RUNS)

rows_6 = []
for k, v in combined_runs.items():
    rows_6.append(
        {
            "Run": v["run_label"],
            "Val AUROC": round(v["val"]["mean_auroc"], 4),
            "Test AUROC": round(v["test"]["mean_auroc"], 4),
            "Test Macro-F1": round(v["test"]["macro_f1"], 4),
            "Test Precision": round(v["test"]["macro_precision"], 4),
            "Test Recall": round(v["test"]["macro_recall"], 4),
            "Delta vs Base Paper (0.812)": f"{v['test']['mean_auroc'] - BASE_PAPER_CHEXPERT_AUROC:+.4f}",
        }
    )
all6_df = pd.DataFrame(rows_6)
print("\n=== Consolidated Part 2 All-Runs Summary (CheXpert) ===")
print(all6_df.to_string(index=False))
all6_df.to_csv(f"{CHECKPOINT_DIR}/part2_chexpert_all6_summary.csv", index=False)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for key, label, color in [
    ("run4_convnext_only", "Run 4: ConvNeXt-Base", "#4c72b0"),
    ("run5_convnext_lbp", "Run 5: + LBP (Cross-Attn)", "#dd8452"),
    ("run6_convnext_edgelog", "Run 6: + Edge/LoG (Cross-Attn)", "#55a868"),
]:
    hist = pd.DataFrame(MODEL2_RUNS[key]["history"])
    axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label=label, color=color)
    axes[1].plot(hist["epoch"], hist["val_auroc"], marker="s", label=label, color=color)

axes[0].set_title("Model 2 Training Loss (CheXpert)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].legend()

axes[1].axhline(BASE_PAPER_CHEXPERT_AUROC, color="crimson", linestyle="--", label="Base Paper (0.812)")
axes[1].set_title("Model 2 Validation Mean AUROC (CheXpert)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean AUROC")
axes[1].legend()
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/part2_model2_chexpert_curves.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=all6_df, x="Run", y="Test AUROC", color="#2b5c8f", ax=ax)
ax.axhline(BASE_PAPER_CHEXPERT_AUROC, color="crimson", linestyle="--", linewidth=2, label="Base Paper (0.812)")
ax.set_ylim(0.5, 1.0)
ax.set_title("Part 2 CheXpert — All Experimental Runs vs. Base Paper", fontweight="bold")
plt.xticks(rotation=20, ha="right")
ax.legend()
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/part2_chexpert_all6_bar.png", dpi=200)
plt.show()